In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)

print("✅ Libraries imported!")
print("💓 Agent 3: Vitals Analyst (Temporal Patterns)")
print("=" * 60)

✅ Libraries imported!
💓 Agent 3: Vitals Analyst (Temporal Patterns)


In [2]:
print("📂 Loading patient splits...")
print("=" * 60)

# Load train/val/test patient IDs
train_patients = pd.read_csv('../../data/processed/train_patients.csv')
val_patients = pd.read_csv('../../data/processed/val_patients.csv')
test_patients = pd.read_csv('../../data/processed/test_patients.csv')

train_patient_ids = train_patients['SUBJECT_ID'].values
val_patient_ids = val_patients['SUBJECT_ID'].values
test_patient_ids = test_patients['SUBJECT_ID'].values

print(f"✅ Patient splits loaded:")
print(f"   Training:   {len(train_patient_ids):,} patients")
print(f"   Validation: {len(val_patient_ids):,} patients")
print(f"   Test:       {len(test_patient_ids):,} patients")

📂 Loading patient splits...
✅ Patient splits loaded:
   Training:   21,816 patients
   Validation: 4,675 patients
   Test:       4,675 patients


In [ ]:
print("📊 Exploring CHARTEVENTS.csv structure...")
print("=" * 60)

# Load first 10,000 rows to understand structure
chart_sample = pd.read_csv('../../data/raw/CHARTEVENTS.csv', nrows=10000)

print(f"✅ Sample loaded: {len(chart_sample):,} rows")
print(f"\nColumns:")
print(chart_sample.columns.tolist())

print(f"\nData types:")
print(chart_sample.dtypes)

print(f"\nFirst few rows:")
print(chart_sample.head())

print(f"\nUnique ITEMID count:")
print(f"   {chart_sample['ITEMID'].nunique():,} different vital sign types in sample")

print(f"\nSample measurements:")
print(chart_sample[['ITEMID', 'VALUE', 'VALUENUM']].head(20))

In [ ]:
print("📖 Loading vital signs dictionary...")
print("=" * 60)

# Load dictionary to understand what each ITEMID means
d_items = pd.read_csv('../../data/raw/D_ITEMS.csv')

print(f"✅ Loaded {len(d_items):,} item definitions")
print(f"\nFirst 20 items:")
print(d_items.head(20))

print(f"\nColumns:")
print(d_items.columns.tolist())

# Check categories
print(f"\nCategories of measurements:")
print(d_items['CATEGORY'].value_counts().head(20))

In [5]:
print("🎯 Selecting important vital signs for ICU diagnosis...")
print("=" * 60)

# KEY VITAL SIGNS for ICU patients
# Based on clinical importance and availability in MIMIC-III

important_vitals = {
    # Cardiovascular
    '220045': 'Heart Rate',
    '220050': 'Arterial Blood Pressure - Systolic',
    '220051': 'Arterial Blood Pressure - Diastolic', 
    '220052': 'Arterial Blood Pressure - Mean',
    '220179': 'Non Invasive Blood Pressure - Systolic',
    '220180': 'Non Invasive Blood Pressure - Diastolic',
    '220181': 'Non Invasive Blood Pressure - Mean',
    
    # Respiratory
    '220210': 'Respiratory Rate',
    '220277': 'SpO2 (Oxygen Saturation)',
    
    # Temperature
    '223761': 'Temperature Fahrenheit',
    '223762': 'Temperature Celsius',
    
    # Other vital signs
    '220339': 'GCS - Glasgow Coma Scale Total',
    '223835': 'FiO2 (Inspired O2 Fraction)',
}

print(f"🎯 Tracking {len(important_vitals)} important vital signs")
print("\nVital signs being tracked:")
for itemid, name in important_vitals.items():
    print(f"  {itemid}: {name}")

🎯 Selecting important vital signs for ICU diagnosis...
🎯 Tracking 13 important vital signs

Vital signs being tracked:
  220045: Heart Rate
  220050: Arterial Blood Pressure - Systolic
  220051: Arterial Blood Pressure - Diastolic
  220052: Arterial Blood Pressure - Mean
  220179: Non Invasive Blood Pressure - Systolic
  220180: Non Invasive Blood Pressure - Diastolic
  220181: Non Invasive Blood Pressure - Mean
  220210: Respiratory Rate
  220277: SpO2 (Oxygen Saturation)
  223761: Temperature Fahrenheit
  223762: Temperature Celsius
  220339: GCS - Glasgow Coma Scale Total
  223835: FiO2 (Inspired O2 Fraction)


In [6]:
print("💓 Loading CHARTEVENTS.csv for training patients...")
print("=" * 60)
print("⏰ WARNING: This will take 20-30 minutes!")
print("   CHARTEVENTS is HUGE (~330 million rows)")
print("   We're filtering to only training patients + important vitals")
print()

# Convert vital codes to integers
important_vital_codes = [int(code) for code in important_vitals.keys()]

# Load in chunks
chunk_size = 1_000_000  # 1 million rows at a time
vital_chunks = []

chunk_counter = 0
total_rows_processed = 0
rows_kept = 0

print(f"📊 Processing CHARTEVENTS in chunks...")
for chunk in tqdm(pd.read_csv('../../data/raw/CHARTEVENTS.csv', chunksize=chunk_size)):
    chunk_counter += 1
    total_rows_processed += len(chunk)
    
    # Filter to:
    # 1. Training patients only
    # 2. Important vital signs only
    # 3. Valid numeric values only
    chunk_filtered = chunk[
        (chunk['SUBJECT_ID'].isin(train_patient_ids)) &
        (chunk['ITEMID'].isin(important_vital_codes)) &
        (chunk['VALUENUM'].notna())
    ]
    
    rows_kept += len(chunk_filtered)
    
    if len(chunk_filtered) > 0:
        vital_chunks.append(chunk_filtered)
    
    # Print progress every 10 chunks
    if chunk_counter % 10 == 0:
        print(f"  Processed {total_rows_processed:,} rows, kept {rows_kept:,} rows ({rows_kept/total_rows_processed*100:.2f}%)")

# Combine all chunks
print("\nCombining chunks...")
vitals_train = pd.concat(vital_chunks, ignore_index=True)

print("=" * 60)
print(f"✅ DONE!")
print(f"   Total rows processed: {total_rows_processed:,}")
print(f"   Rows kept: {len(vitals_train):,}")
print(f"   Reduction: {(1 - len(vitals_train)/total_rows_processed)*100:.1f}%")
print(f"   Unique patients: {vitals_train['SUBJECT_ID'].nunique():,}")
print(f"   Unique vital types: {vitals_train['ITEMID'].nunique():,}")

💓 Loading CHARTEVENTS.csv for training patients...
⏰ WARNING: This will take 20-30 minutes!
   CHARTEVENTS is HUGE (~330 million rows)
   We're filtering to only training patients + important vitals

📊 Processing CHARTEVENTS in chunks...


10it [00:11,  1.17s/it]

  Processed 10,000,000 rows, kept 3,038,521 rows (30.39%)


20it [00:22,  1.10s/it]

  Processed 20,000,000 rows, kept 6,393,481 rows (31.97%)


...

  Processed 320,000,000 rows, kept 11,273,963 rows (3.52%)


330it [06:46,  1.25s/it]

  Processed 330,000,000 rows, kept 11,273,963 rows (3.42%)


331it [06:47,  1.23s/it]



Combining chunks...
✅ DONE!
   Total rows processed: 330,712,483
   Rows kept: 11,273,963
   Reduction: 96.6%
   Unique patients: 10,167
   Unique vital types: 13


In [8]:
print("💾 Saving filtered vitals...")
print("=" * 60)

# Drop VALUE column (we only need VALUENUM which is clean numeric)
# VALUE has mixed types (strings like "100", "Normal", etc.)
vitals_train_clean = vitals_train.drop(columns=['VALUE'], errors='ignore')

# Save to parquet
vitals_train_clean.to_parquet('../../data/processed/vitals_train_filtered.parquet')

print(f"✅ Saved {len(vitals_train_clean):,} vital sign measurements")
print(f"   File: data/processed/vitals_train_filtered.parquet")
print(f"   Size: ~{vitals_train_clean.memory_usage(deep=True).sum() / 1024**2:.1f} MB in memory")

# Also show what columns we kept
print(f"\nColumns saved:")
print(vitals_train_clean.columns.tolist())

💾 Saving filtered vitals...
✅ Saved 11,273,963 vital sign measurements
   File: data/processed/vitals_train_filtered.parquet
   Size: ~3415.8 MB in memory

Columns saved:
['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ITEMID', 'CHARTTIME', 'STORETIME', 'CGID', 'VALUENUM', 'VALUEUOM', 'WARNING', 'ERROR', 'RESULTSTATUS', 'STOPPED']


In [ ]:
print("🔍 Examining filtered vital signs data...")
print("=" * 60)

print(f"First 10 rows:")
print(vitals_train.head(10))

print(f"\nData types:")
print(vitals_train.dtypes)

print(f"\nVital sign distribution:")
vital_counts = vitals_train['ITEMID'].value_counts()
print(vital_counts)

print(f"\nStatistics per vital sign:")
for itemid, name in important_vitals.items():
    itemid_int = int(itemid)
    if itemid_int in vital_counts.index:
        count = vital_counts[itemid_int]
        values = vitals_train[vitals_train['ITEMID'] == itemid_int]['VALUENUM']
        print(f"\n{name} ({itemid}):")
        print(f"  Count: {count:,}")
        print(f"  Mean: {values.mean():.2f}")
        print(f"  Min: {values.min():.2f}")
        print(f"  Max: {values.max():.2f}")

In [10]:
print("⏱️  Creating temporal features per patient...")
print("=" * 60)

# For each patient, compute statistics over time
print(f"Aggregating vitals for {vitals_train['SUBJECT_ID'].nunique():,} patients...")

agg_features = []

for patient_id in tqdm(train_patient_ids, desc="Processing patients"):
    patient_vitals = vitals_train[vitals_train['SUBJECT_ID'] == patient_id]
    
    if len(patient_vitals) == 0:
        # Patient has no vitals - will be handled later
        features = {'SUBJECT_ID': patient_id}
        for itemid in important_vital_codes:
            features[f'vital_{itemid}_mean'] = np.nan
            features[f'vital_{itemid}_min'] = np.nan
            features[f'vital_{itemid}_max'] = np.nan
            features[f'vital_{itemid}_std'] = np.nan
            features[f'vital_{itemid}_count'] = 0
        agg_features.append(features)
        continue
    
    features = {'SUBJECT_ID': patient_id}
    
    for itemid in important_vital_codes:
        vital_values = patient_vitals[patient_vitals['ITEMID'] == itemid]['VALUENUM'].dropna()
        
        if len(vital_values) > 0:
            features[f'vital_{itemid}_mean'] = vital_values.mean()
            features[f'vital_{itemid}_min'] = vital_values.min()
            features[f'vital_{itemid}_max'] = vital_values.max()
            features[f'vital_{itemid}_std'] = vital_values.std() if len(vital_values) > 1 else 0
            features[f'vital_{itemid}_count'] = len(vital_values)
        else:
            features[f'vital_{itemid}_mean'] = np.nan
            features[f'vital_{itemid}_min'] = np.nan
            features[f'vital_{itemid}_max'] = np.nan
            features[f'vital_{itemid}_std'] = np.nan
            features[f'vital_{itemid}_count'] = 0
    
    agg_features.append(features)

vital_features_train = pd.DataFrame(agg_features)

print(f"\n✅ Aggregation complete!")
print(f"   Patients: {len(vital_features_train):,}")
print(f"   Features per patient: {vital_features_train.shape[1] - 1}")

⏱️  Creating temporal features per patient...
Aggregating vitals for 10,167 patients...


Processing patients: 100%|███████████████████████████████████████████████████████| 21816/21816 [05:54<00:00, 61.56it/s]



✅ Aggregation complete!
   Patients: 21,816
   Features per patient: 65


In [11]:
print("📊 Analyzing vital signs coverage...")
print("=" * 60)

# Check how many patients have each vital
print(f"Vital sign coverage across {len(vital_features_train):,} training patients:\n")

for itemid, name in important_vitals.items():
    count_col = f'vital_{itemid}_count'
    if count_col in vital_features_train.columns:
        patients_with_vital = (vital_features_train[count_col] > 0).sum()
        coverage = patients_with_vital / len(vital_features_train) * 100
        print(f"{name:50s}: {patients_with_vital:5,} patients ({coverage:5.1f}%)")

# Overall coverage
print(f"\nPatients with ANY vitals:")
has_any_vital = (vital_features_train.iloc[:, 1:].sum(axis=1) > 0).sum()
print(f"   {has_any_vital:,} / {len(vital_features_train):,} ({has_any_vital/len(vital_features_train)*100:.1f}%)")

📊 Analyzing vital signs coverage...
Vital sign coverage across 21,816 training patients:

Heart Rate                                        : 10,163 patients ( 46.6%)
Arterial Blood Pressure - Systolic                : 5,375 patients ( 24.6%)
Arterial Blood Pressure - Diastolic               : 5,374 patients ( 24.6%)
Arterial Blood Pressure - Mean                    : 5,431 patients ( 24.9%)
Non Invasive Blood Pressure - Systolic            : 10,063 patients ( 46.1%)
Non Invasive Blood Pressure - Diastolic           : 10,063 patients ( 46.1%)
Non Invasive Blood Pressure - Mean                : 10,058 patients ( 46.1%)
Respiratory Rate                                  : 10,157 patients ( 46.6%)
SpO2 (Oxygen Saturation)                          : 10,152 patients ( 46.5%)
Temperature Fahrenheit                            : 10,034 patients ( 46.0%)
Temperature Celsius                               : 1,447 patients (  6.6%)
GCS - Glasgow Coma Scale Total                    : 5,708 patients 